<h2><a href="https://leetcode.com/problems/min-stack">155. Min Stack</a></h2><h3>Medium</h3><hr><p>Design a stack that supports push, pop, top, and retrieving the minimum element in constant time.</p>

<p>Implement the <code>MinStack</code> class:</p>

<ul>
	<li><code>MinStack()</code> initializes the stack object.</li>
	<li><code>void push(int val)</code> pushes the element <code>val</code> onto the stack.</li>
	<li><code>void pop()</code> removes the element on the top of the stack.</li>
	<li><code>int top()</code> gets the top element of the stack.</li>
	<li><code>int getMin()</code> retrieves the minimum element in the stack.</li>
</ul>

<p>You must implement a solution with <code>O(1)</code> time complexity for each function.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>

<pre>
<strong>Input</strong>
[&quot;MinStack&quot;,&quot;push&quot;,&quot;push&quot;,&quot;push&quot;,&quot;getMin&quot;,&quot;pop&quot;,&quot;top&quot;,&quot;getMin&quot;]
[[],[-2],[0],[-3],[],[],[],[]]

<strong>Output</strong>
[null,null,null,null,-3,null,0,-2]

<strong>Explanation</strong>
MinStack minStack = new MinStack();
minStack.push(-2);
minStack.push(0);
minStack.push(-3);
minStack.getMin(); // return -3
minStack.pop();
minStack.top();    // return 0
minStack.getMin(); // return -2
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>-2<sup>31</sup> &lt;= val &lt;= 2<sup>31</sup> - 1</code></li>
	<li>Methods <code>pop</code>, <code>top</code> and <code>getMin</code> operations will always be called on <strong>non-empty</strong> stacks.</li>
	<li>At most <code>3 * 10<sup>4</sup></code> calls will be made to <code>push</code>, <code>pop</code>, <code>top</code>, and <code>getMin</code>.</li>
</ul>


# Min Stack Implementation

## 📚 Problem Overview

Design a stack that supports:
- `push(val)`: Add element to top
- `pop()`: Remove top element
- `top()`: Get top element
- `getMin()`: Get minimum element

**All operations must be O(1) time complexity.**

The challenge is maintaining the minimum efficiently without scanning the entire stack each time.

---

## 🧠 Solution 1: Two-Stack Approach

### Intuition and Approach

Use **two separate stacks**:
- `stack`: Stores all elements (normal stack behavior)
- `m`: Stores minimum values (monotonic stack - non-increasing)

**Key Insight:**
- When pushing, only add to `m` if the new value is ≤ current minimum
- When popping, only remove from `m` if the popped value equals the current minimum
- This ensures `m` always contains the minimum path to the current stack state

**Why this works:**
- `m` maintains a decreasing sequence of minimums
- The top of `m` is always the current minimum
- Popping removes minimums only when necessary

### Code Explanation

```python
from collections import deque
class MinStack:

    def __init__(self):
        self.m = deque()        # Minimum stack
        self.stack = deque()    # Main stack
```
- Two deques: `m` tracks minimums, `stack` holds actual values
- `deque` chosen for O(1) append/pop operations

```python
    def push(self, val: int) -> None:
        if len(self.m) == 0 or val <= self.m[-1]:
            self.m.append(val)
        self.stack.append(val)
```
- **Always push to main stack**
- **Conditionally push to min stack**: Only if `val` is ≤ current minimum (`self.m[-1]`)
- Why `<=`? Handles duplicate minimums correctly (e.g., multiple -2's)
- If `m` is empty, push first value as initial minimum

```python
    def pop(self) -> None:
        if len(self.stack) == 0:
            return
        if self.stack[-1] == self.m[-1]:
            self.m.pop()
        return self.stack.pop()
```
- **Always pop from main stack**
- **Conditionally pop from min stack**: Only if the value being popped equals the current minimum
- This maintains the invariant that `m` contains all minimums up to current state
- Why check equality? Because we only pushed when `val <= m[-1]`, so we need to remove when that specific value is popped

```python
    def top(self) -> int:
        if len(self.stack) == 0:
            return
        return self.stack[-1]
```
- Return top of main stack
- Guard against empty stack (though problem guarantees non-empty calls)

```python
    def getMin(self) -> int:
        return self.m[-1]
```
- Return top of minimum stack (current minimum)
- Always valid since `m` mirrors the minimum history

---

### 🧪 Dry Run: Two-Stack Approach

```
Operations: push(-2), push(0), push(-3), getMin(), pop(), top(), getMin()
```

**Initial:** stack=[], m=[]

1. **push(-2)**
   - len(m) == 0 → push -2 to m
   - push -2 to stack
   - **Result:** stack=[-2], m=[-2]

2. **push(0)**
   - 0 > m[-1]=-2 → don't push to m
   - push 0 to stack
   - **Result:** stack=[-2, 0], m=[-2]

3. **push(-3)**
   - -3 < m[-1]=-2 → push -3 to m
   - push -3 to stack
   - **Result:** stack=[-2, 0, -3], m=[-2, -3]

4. **getMin()**
   - return m[-1] = -3 ✓

5. **pop()**
   - stack[-1] = -3 == m[-1] = -3 → pop m (removes -3)
   - pop stack (removes -3)
   - **Result:** stack=[-2, 0], m=[-2]

6. **top()**
   - return stack[-1] = 0 ✓

7. **getMin()**
   - return m[-1] = -2 ✓

---

## 🧠 Solution 2: Single-Stack with Pairs

### Intuition and Approach

Use **one stack** where each element is a **pair: [value, current_minimum]**

**Key Insight:**
- Store both the actual value and the minimum up to that point
- When pushing, compute new minimum as `min(current_min, new_value)`
- This creates a "minimum history" embedded in each element

**Why this works:**
- Each stack element knows the minimum from start to its position
- No separate data structure needed
- Popping automatically restores previous minimum state

### Code Explanation

```python
from collections import deque
class MinStack:

    def __init__(self):
        self.stack = deque()    # Single stack with pairs
```

```python
    def push(self, val: int) -> None:
        if len(self.stack) == 0:
            self.stack.append([val, val])  # First element: [value, min_so_far]
        else:
            m = min(self.stack[-1][1], val)  # Current min vs new value
            self.stack.append([val, m])     # Store [value, updated_min]
```
- **Empty stack:** Push `[val, val]` (value is its own minimum)
- **Non-empty:** Compute new minimum = min(previous_minimum, val)
- Store as `[val, new_minimum]`
- This propagates the minimum forward

```python
    def pop(self) -> None:
        if len(self.stack) == 0:
            return
        return self.stack.pop()[0]  # Return just the value, discard min
```
- Pop the pair, return only the value (ignore the minimum)
- The minimum information is lost, but that's OK since it's embedded in remaining elements

```python
    def top(self) -> int:
        if len(self.stack) == 0:
            return
        return self.stack[-1][0]  # Return value part of top pair
```

```python
    def getMin(self) -> int:
        if len(self.stack) == 0:
            return 0  # Guard (though problem guarantees non-empty)
        return self.stack[-1][1]  # Return minimum part of top pair
```
- The minimum is always the second element of the top pair

---

### 🧪 Dry Run: Single-Stack Approach

```
Operations: push(-2), push(0), push(-3), getMin(), pop(), top(), getMin()
```

**Initial:** stack=[]

1. **push(-2)**
   - len(stack) == 0 → append([-2, -2])
   - **Result:** stack=[[-2, -2]]

2. **push(0)**
   - m = min(stack[-1][1]=-2, 0) = -2
   - append([0, -2])
   - **Result:** stack=[[-2, -2], [0, -2]]

3. **push(-3)**
   - m = min(stack[-1][1]=-2, -3) = -3
   - append([-3, -3])
   - **Result:** stack=[[-2, -2], [0, -2], [-3, -3]]

4. **getMin()**
   - return stack[-1][1] = -3 ✓

5. **pop()**
   - pop() returns [-3, -3][0] = -3
   - **Result:** stack=[[-2, -2], [0, -2]]

6. **top()**
   - return stack[-1][0] = 0 ✓

7. **getMin()**
   - return stack[-1][1] = -2 ✓

---

## ⚠️ Edge Cases

### Case 1: Empty Stack Operations
- **Problem guarantee:** Methods called on non-empty stacks, but code has guards
- `pop()`/`top()`/`getMin()` return `None` or `0` if empty

### Case 2: Single Element
```python
push(5)
getMin() → 5
pop() → stack=[]
getMin() → 0 (guard)
```

### Case 3: Duplicate Minimums
```python
push(-2), push(-2), push(-2)
getMin() → -2
pop() → still -2 (multiple -2's)
pop() → still -2
pop() → stack empty
```
**Solution 1:** Only pushes to `m` on `<=`, handles duplicates correctly
**Solution 2:** Each push updates minimum, but popping preserves history

### Case 4: Increasing Values
```python
push(1), push(2), push(3)
getMin() → 1 (always the global minimum)
```

### Case 5: Decreasing Values
```python
push(3), push(2), push(1)
getMin() → 1 (updates with each smaller value)
```

### Case 6: Negative Numbers
```python
push(-10), push(-5), push(-15)
getMin() → -15
```
**Handled:** No issues with negative values

### Case 7: Zero Values
```python
push(0), push(5), push(-1)
getMin() → -1
```
**Handled:** Zero is valid value

### Case 8: Large Values (Constraints: -2^31 to 2^31-1)
- Both solutions handle full int32 range
- No overflow issues in Python

### Case 9: Maximum Operations (3*10^4)
- Both solutions scale to 30,000 operations efficiently

---

## ⏱️ Time & Space Complexity

### Solution 1: Two Stacks

| Operation | Best | Average | Worst | Explanation |
|-----------|------|---------|-------|-------------|
| `push`    | O(1) | O(1) | O(1) | Conditional append to both stacks |
| `pop`     | O(1) | O(1) | O(1) | Conditional pop from min stack |
| `top`     | O(1) | O(1) | O(1) | Access main stack top |
| `getMin`  | O(1) | O(1) | O(1) | Access min stack top |

**Space:** O(n) - Main stack O(n), min stack O(n) in worst case (all decreasing)

### Solution 2: Single Stack with Pairs

| Operation | Best | Average | Worst | Explanation |
|-----------|------|---------|-------|-------------|
| `push`    | O(1) | O(1) | O(1) | Compute min and append pair |
| `pop`     | O(1) | O(1) | O(1) | Pop pair |
| `top`     | O(1) | O(1) | O(1) | Access pair[0] |
| `getMin`  | O(1) | O(1) | O(1) | Access pair[1] |

**Space:** O(n) - Each element stores value + minimum (2x space)

---

## 🔄 Comparison

| Aspect | Two Stacks | Single Stack |
|--------|-----------|--------------|
| **Space** | O(n) | O(n) (but 2x) |
| **Clarity** | Clear separation | Embedded logic |
| **Duplicates** | Handles with `<=` | Automatic |
| **Implementation** | Simpler logic | More complex pairs |
| **Memory** | Efficient | Slightly more overhead |

**Both achieve O(1) time for all operations as required.**

---

## 📝 Key Insights

- **Two-stack approach** is more intuitive and commonly taught
- **Single-stack approach** saves space conceptually but uses more memory per element
- **Critical detail:** Handling duplicate minimums requires careful logic
- **Trade-off:** Space vs. conceptual simplicity
- **Both valid:** Choose based on interview context or preference

The problem demonstrates how to augment data structures to support additional operations efficiently, a common design pattern in system design and interviews.

In [32]:
from collections import deque
class MinStack:

    def __init__(self):
        self.m = deque()
        self.stack = deque()

    def push(self, val: int) -> None:
        if len(self.m) == 0 or val <= self.m[-1]:
            self.m.append(val)
        self.stack.append(val)

    def pop(self) -> None:
        if len(self.stack) == 0:
            return
        if self.stack[-1] == self.m[-1]:
            self.m.pop()
        return self.stack.pop()

    def top(self) -> int:
        if len(self.stack) == 0:
            return
        return self.stack[-1]

    def getMin(self) -> int:
        return self.m[-1]

# Your MinStack object will be instantiated and called as such:
minStack = MinStack()
print(minStack.push(-2))
print(minStack.push(0))
print(minStack.push(-3))
print(minStack.getMin())
print(minStack.pop())
print(minStack.top())
print(minStack.getMin())

None
None
None
-3
-3
0
-2


In [31]:
from collections import deque
class MinStack:

    def __init__(self):
        self.stack = deque()

    def push(self, val: int) -> None:
        if len(self.stack) == 0:
            self.stack.append([val,val])
        else:
            m = min(self.stack[-1][1],val)
            self.stack.append([val,m])

    def pop(self) -> None:
        if len(self.stack) == 0:
            return
        return self.stack.pop()[0]

    def top(self) -> int:
        if len(self.stack) == 0:
            return
        return self.stack[-1][0]

    def getMin(self) -> int:
        if len(self.stack) == 0:
            return 0
        return self.stack[-1][1]

# Your MinStack object will be instantiated and called as such:
minStack = MinStack()
print(minStack.push(-2))
print(minStack.push(0))
print(minStack.push(-3))
print(minStack.getMin())
print(minStack.pop())
print(minStack.top())
print(minStack.getMin())

None
None
None
-3
-3
0
-2
